# Extractor de Estadísticas del Catastro Inmobiliario
@jcricocordoba

### Presentación del Cuaderno

Este cuaderno de Google Colab ha sido diseñado para **automatizar y simplificar** la descarga de datos estadísticos de la **Dirección General del Catastro de España**.

El principal problema que resuelve es la **extracción manual**, un proceso que consume mucho tiempo, es repetitivo y propenso a errores. Con este *script*, puedes obtener un conjunto completo de datos para una provincia y años específicos con solo unos pocos clics, permitiéndote dedicar tu tiempo a tareas más importantes como el análisis de la información.

---

### Objetivos y Funcionamiento

El flujo de trabajo se divide en los siguientes pasos:
- [ ] **1. Configurar el entorno**: La primera celda instala todas las herramientas necesarias.
- [ ] **2. Seleccionar los parámetros**: Unos menús interactivos te permitirán elegir la provincia y los años para los datos.
- [ ] **3. Ejecutar la extracción**: Un botón iniciará el proceso de *scraping* automático.
- [ ] **4. Descargar los resultados**: Finalmente, otro botón comprimirá todos los ficheros `.csv` en un único archivo `.zip` para que lo puedas descargar.

### Bloques de Datos Extraídos

El *script* está preparado para extraer los siguientes bloques de información de ámbito municipal y/o provincial:

| Bloque | Título del Bloque | Descripción de los Datos |
| :--- | :--- | :--- |
| **B1** | `Catastro Urbano` | Variables generales, parcelas, superficies, usos y valor catastral. |
| **B2** | `Urbana Anuales` | Inmuebles, superficie media y valor catastral medio por distrito. |
| **B4** | `Titularidad` | Número y tipo de titulares por municipio. |
| **B5** | `Catastro Rústico` | Variables generales y porcentajes por tipo de cultivo. |
| **B8** | `IBI` | Bases imponibles y cuotas líquidas para IBI urbano, rústico y BICE. |
| **B9** | `Catastro BICE` | Datos de Bienes Inmuebles de Características Especiales. |
| **B12**| `Ordenanzas Fiscales`| Tipos de gravamen y recargos sobre inmuebles desocupados. |

## Celda 1: Instalación de Dependencias e Importación de Módulos

### Entorno de Ejecución

Esta celda inicial configura el entorno de Google Colab para que el script funcione correctamente. Las acciones que realiza son:

- **Instalar librerías de Python**:
    - `playwright`: *La herramienta clave para la automatización web*. Permite controlar un navegador para simular la interacción humana (clics, selecciones, etc.).
    - `pandas`: *Fundamental para la manipulación y el análisis de datos*. Se usará para estructurar la información en tablas y guardarla en formato `.csv`.
    - `beautifulsoup4`: *Una librería especializada en web scraping*. Facilita la búsqueda y extracción de datos de las tablas HTML.
    - `ipywidgets`: *Permite crear elementos interactivos* como botones y menús desplegables en el notebook.
    - `lxml`: *Un analizador de HTML de alto rendimiento* que `BeautifulSoup` utiliza internamente para ser más rápido y eficiente.

- **Instalar el navegador**: El comando `!playwright install chromium` descarga una instancia del navegador **Chromium**, que será la que `playwright` controlará de forma invisible (*headless*).

- **Importar módulos**: Se importan todas las librerías instaladas junto con otras herramientas estándar de Python (`os`, `re`, `unicodedata`, etc.).

> **Nota importante**: `nest_asyncio.apply()` es una línea crucial en Colab. `Playwright` funciona de manera asíncrona, y esta función permite que coexista sin conflictos con el bucle de eventos que Colab ya tiene en ejecución.


In [1]:
# -*- coding: utf-8 -*- # Define la codificación de caracteres del archivo.

# =============================================================================
# Celda 1: Configuración del entorno e importación de librerías
# =============================================================================

# --- Importaciones iniciales para la configuración ---
from IPython.display import display, clear_output  # Funciones para controlar la salida de la celda.
try:
    # Intenta importar 'files' para la descarga de archivos en Google Colab.
    from google.colab import files
except ImportError:
    # Si no se está en Colab, se asigna None para evitar errores posteriores.
    files = None

# --- Instalación de dependencias ---
print("Instalando dependencias... (puede tardar un momento)")

# Instala las librerías necesarias de forma silenciosa (-q).
# playwright: para automatización web.
# pandas: para manipulación de datos.
# beautifulsoup4: para parsear HTML.
# ipywidgets: para crear widgets interactivos.
!pip install -q playwright pandas beautifulsoup4 ipywidgets lxml nest_asyncio

# Descarga el navegador (Chromium) que Playwright necesita para operar.
!playwright install chromium
!playwright install-deps chromium
clear_output()  # Limpia la salida de la instalación para mantener el notebook limpio.

# --- Importaciones principales del script ---
import os
import re
import shutil
import unicodedata
import json
import asyncio
import time
from typing import Optional, Dict, Tuple  # Para anotaciones de tipo que mejoran la legibilidad.

# Librerías de terceros
import pandas as pd
import requests                                # Para realizar peticiones HTTP.
import ipywidgets as widgets                   # Para crear interfaces de usuario interactivas.
from bs4 import BeautifulSoup                  # Para extraer datos de archivos HTML y XML.
from playwright.async_api import async_playwright  # Para controlar el navegador de forma asíncrona.
import nest_asyncio                            # Permite que la programación asíncrona funcione en Jupyter/Colab.

# Aplica un parche para evitar errores de "event loop is already running" en entornos de notebook.
nest_asyncio.apply()

print("✅ Entorno y módulos listos.")
print("-" * 50)

✅ Entorno y módulos listos.
--------------------------------------------------


## Celda 2: Constantes y Funciones para Construir URLs

### Lógica de Generación de Enlaces

En esta celda se define el "mapa" para acceder a cada conjunto de datos del Catastro. Dado que las URLs cambian según el **año** y la **provincia**, este código se encarga de construirlas dinámicamente.

- **`BASE_B*`**: Son las plantillas de las URLs para cada bloque estadístico. Contienen marcadores de posición como `{year}` y `{file}` que serán reemplazados.

- **`PREFIJOS_B*`**: Son diccionarios que asocian un nombre descriptivo (ej: `"01a_variables_catastro"`) con el código de archivo que usa el Catastro (ej: `"041"`).
    - Para los bloques 9 y 12, la tupla `(prefijo, con_prov)` permite gestionar datos que a veces son nacionales y otras provinciales. Si `con_prov` es **True**, se añade el código de provincia; si es **False**, no.

- **`build_urls_b*()`**: Cada una de estas funciones toma un `año` y un `código de provincia` y genera un diccionario con todas las URLs finales para un bloque, listas para ser procesadas.

- **`_prov()`**: Es una pequeña función de ayuda que asegura que el código de provincia siempre tenga un formato de **dos dígitos** (ej: `5` se convierte en `05`).


In [2]:
# =============================================================================
# Celda 2: Constantes y funciones para construir URLs
# =============================================================================

# URLs base para cada bloque de datos del Catastro.
# Contienen marcadores de posición {year} y {file} que se rellenarán dinámicamente.
BASE_B1  = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/catastro/urbano/&file={file}.px&type=pcaxis&L=0"
BASE_B2  = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/catastro/distrito/&file={file}.px&type=pcaxis&L=0"
BASE_B4  = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/catastro/titulares/&file={file}.px&type=pcaxis&L=0"
BASE_B5  = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/catastro/rustico/&file={file}.px&type=pcaxis&L=0"
BASE_B8  = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/ibi/&file={file}.px&type=pcaxis&L=0"
BASE_B9  = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/catastro/bice/&file={file}.px&type=pcaxis&L=0"
BASE_B12 = "https://www.catastro.hacienda.gob.es/jaxi/tabla.do?path=/est{year}/ordenanzasfiscales/&file={file}.px&type=pcaxis&L=0"

# --- Mapas de nombres de tabla a códigos de fichero ---
# Asocian un nombre descriptivo y legible (clave) al código de fichero real (valor).

PREFIJOS_B1 = {
    "01a_variables_catastro": "041", "01b_parcelas_edificadas_propiedad": "043",
    "01c_parcelas_por_superficie": "044", "01d_superficies_parcelas_urbanas": "045",
    "01e_bienes_inmuebles_usos": "URAO42", "01f_valor_catastral_usos": "URBO42"
}
# Regla B2: el fichero se compone como ANU{year}_{n}_{cp}
PREFIJOS_B2 = {
    "02a_inmuebles_distrito_uso": "1", "02b_superficie_media_distrito_uso": "2",
    "02c_valor_catastral_medio_distrito_uso": "3"
}
PREFIJOS_B4 = { "04a_titulares": "040" }
PREFIJOS_B5 = { "05a_rust_variables": "041", "05b_rust_porc_tipo_cultivo": "042" }
PREFIJOS_B8 = { "08a_ibi_urbano": "U040", "08b_ibi_rustico": "R040", "08c_ibi_bice": "B040" }

# Regla B9 y B12: el valor es una tupla (código, booleano) donde el booleano
# indica si el fichero final debe incluir el código de provincia.
PREFIJOS_B9 = {
    "09a6_huertos_solares": ("A6", True), "09b_presas_embalses": ("B", True),
    "09a1_produc_energ_electrica": ("A1", False), "09a2_gas": ("A2", False),
    "09a3_refino_petroleo": ("A3", False), "09a4_central_nuclear": ("A4", False),
    "09a5_parque_eolico": ("A5", False), "09c_carreteras_peaje": ("C", False),
    "09d1_aeropuertos": ("D1", False), "09d2_puertos_comerciales": ("D2", False)
}
PREFIJOS_B12 = {
    "12a_of_tipo_gravamen": ("041", True), "12b_of_entidad": ("042", True),
    "12c_of_tipos_diferenciados": ("04301", False), "12d_of_recargo_bi_desocup": ("04401", False)
}

# --- Funciones constructoras de URLs ---

def _prov(p: int | str) -> str:
    """Normaliza y valida un código de provincia.

    Args:
        p: El código de provincia como número o cadena.

    Returns:
        El código de provincia como una cadena de 2 dígitos (e.g., '01', '28').

    Raises:
        ValueError: Si el código no está en el rango [1, 52].
    """
    p_int = int(p)
    if not (1 <= p_int <= 52):
        raise ValueError("El código de provincia debe estar entre 1 y 52")
    return f"{p_int:02d}"

# REFACTORIZACIÓN: Función genérica para B1, B4, B5, B8
def build_urls_simple_prefix(year: int, cp: str, base_url: str, prefijos: dict) -> dict:
    """Construye URLs para bloques que siguen la regla: file = {prefijo}{cp}."""
    return {nombre: base_url.format(year=year, file=f"{prefijo}{cp}")
            for nombre, prefijo in prefijos.items()}

# B2 tiene una regla de construcción de 'file' única.
def build_urls_b2(year: int, cp: str) -> dict:
    """Construye URLs para B2. Regla: file = ANU{year}_{n}_{cp}."""
    return {nombre: BASE_B2.format(year=year, file=f"ANU{year}_{n}_{cp}")
            for nombre, n in PREFIJOS_B2.items()}

# B9 tiene una lógica condicional para añadir el código de provincia.
def build_urls_b9(year: int, cp: str) -> dict:
    """Construye URLs para B9. Regla: file = {prefijo}{cp} o BICE{year}_{prefijo}."""
    urls = {}
    for nombre, (prefijo, con_prov) in PREFIJOS_B9.items():
        file_code = f"{prefijo}{cp}" if con_prov else f"BICE{year}_{prefijo}"
        urls[nombre] = BASE_B9.format(year=year, file=file_code)
    return urls

# B12 también tiene lógica condicional.
def build_urls_b12(year: int, cp: str) -> dict:
    """Construye URLs para B12. Regla: file = {prefijo}{cp} o solo {prefijo}."""
    urls = {}
    for nombre, (prefijo, con_prov) in PREFIJOS_B12.items():
        file_code = f"{prefijo}{cp}" if con_prov else prefijo
        urls[nombre] = BASE_B12.format(year=year, file=file_code)
    return urls

print("Constantes y funciones para generar URLs de todos los bloques listas.")
print("-" * 50)

Constantes y funciones para generar URLs de todos los bloques listas.
--------------------------------------------------


## Celda 3: Widgets Interactivos de Selección

### Interfaz de Usuario

Para que el script sea reutilizable y fácil de usar, esta celda crea un pequeño formulario interactivo. En lugar de modificar el código, el usuario puede simplemente seleccionar la provincia y los años que le interesan.

- **Función `extraer_opciones_catastro`**: Esta función hace que el notebook sea **"inteligente"**. Se conecta a las páginas de estadísticas del Catastro y **extrae en tiempo real** las opciones disponibles en los menús de la propia web, garantizando que los datos estén siempre actualizados.

- **Extracción de Opciones**: Se llama a la función anterior para obtener las listas de años y provincias disponibles para los diferentes bloques.

- **Determinación de Valores por Defecto**: Para mayor comodidad, los menús se preseleccionan con los valores más lógicos: el **año más reciente** y la provincia de **Córdoba (`14`)**.

- **Creación de los `widgets.Dropdown`**: Se crean los menús desplegables utilizando las opciones extraídas y los valores por defecto, con descripciones claras para el usuario.

In [3]:
# =============================================================================
# Celda 3: Widgets y descubrimiento de opciones desde la web
# =============================================================================

def extraer_opciones_catastro(url: str, id_select_anio: str, id_select_prov: Optional[str]) -> Tuple[Dict, Dict]:
    """Descarga una página del Catastro y extrae las opciones de los menús desplegables.

    Realiza una petición HTTP a la URL proporcionada, parsea el HTML y busca
    los elementos <select> de año y provincia por sus IDs para extraer sus
    valores y textos.

    Args:
        url: La URL de la página a analizar.
        id_select_anio: El ID del <select> que contiene los años.
        id_select_prov: El ID del <select> de provincias. Si es None, no se busca.

    Returns:
        Una tupla con dos diccionarios:
        - El primero mapea el texto del año a su valor (e.g., {'2023': '2023'}).
        - El segundo mapea el nombre de la provincia a su código normalizado de 2 dígitos.
    """
    print(f"Obteniendo opciones de {url}...")
    try:
        r = requests.get(url, timeout=20)  # Petición HTTP con timeout de 20s.
        r.raise_for_status()  # Lanza una excepción para códigos de error (4xx o 5xx).
    except requests.RequestException as e:
        print(f"⚠️ No se pudo obtener el HTML de {url}. Error: {e}")
        return {}, {}

    soup = BeautifulSoup(r.content, 'html.parser')

    # Extraer opciones de año
    sel_year = soup.find('select', {'id': id_select_anio})
    opciones_anio = {opt.text.strip(): opt.get('value') for opt in sel_year.find_all('option') if opt.get('value')} if sel_year else {}

    # Extraer opciones de provincia, si se especifica un ID
    opciones_prov = {}
    if id_select_prov:
        sel_prov = soup.find('select', {'id': id_select_prov})
        if sel_prov:
            for option in sel_prov.find_all('option'):
                val = option.get('value', '')
                # Filtra valores no deseados (vacíos, 'Todas', etc.) y normaliza el código.
                if val and '000' not in val and len(val) >= 2:
                    # La lógica extrae los últimos 2 dígitos si el valor es complejo.
                    codigo = val if val.isdigit() and len(val) <= 2 else val[-2:]
                    if codigo.isdigit():
                        opciones_prov[option.text.strip()] = f"{int(codigo):02d}"

    print("✅ Opciones obtenidas.")
    return opciones_anio, opciones_prov

# --- Configuración y obtención de opciones ---

# Páginas índice por bloque desde donde se extraen las opciones.
URLS_PAGINAS = {
    "b1|b4|b5|b9": "https://www.catastro.hacienda.gob.es/esp/estadistica_1.asp",
    "b2": "https://www.catastro.hacienda.gob.es/esp/estadistica_4.asp",
    "b8": "https://www.catastro.hacienda.gob.es/esp/estadistica_6.asp"
}

# Obtiene dinámicamente los años y provincias disponibles para cada grupo de bloques.
opciones_year_b1, opciones_provincia = extraer_opciones_catastro(URLS_PAGINAS["b1|b4|b5|b9"], "urbano", "select_URB_4")
opciones_year_b2, _ = extraer_opciones_catastro(URLS_PAGINAS["b2"], "anuarios", None)  # B2 no tiene selector de provincia en esta página.
opciones_year_b8, _ = extraer_opciones_catastro(URLS_PAGINAS["b8"], "ibi", None)      # B8 tampoco.

# --- Creación de los Widgets interactivos ---

# Selecciones por defecto: el último año disponible y la provincia de Córdoba ('14') si existe.
default_year_b1 = max(opciones_year_b1.keys()) if opciones_year_b1 else None
default_year_b2 = max(opciones_year_b2.keys()) if opciones_year_b2 else None
default_year_b8 = max(opciones_year_b8.keys()) if opciones_year_b8 else None
default_prov = '14' if '14' in opciones_provincia.values() else (list(opciones_provincia.values())[0] if opciones_provincia else None)

# Se crean los menús desplegables (Dropdowns) que se mostrarán al usuario.
# Las descripciones indican a qué bloques de datos afecta cada selector.
selector_provincia = widgets.Dropdown(options=opciones_provincia, description='Provincia:', value=default_prov, style={'description_width': 'initial'})
selector_year_b1 = widgets.Dropdown(options=opciones_year_b1, description='[Urbano/Rústico/BICE]:', value=default_year_b1, style={'description_width': 'initial'})
selector_year_b2 = widgets.Dropdown(options=opciones_year_b2, description='[Distrito/Uso]:', value=default_year_b2, style={'description_width': 'initial'})
selector_year_b8 = widgets.Dropdown(options=opciones_year_b8, description='[IBI]:', value=default_year_b8, style={'description_width': 'initial'})

# Muestra los widgets en la salida de la celda.
display(selector_provincia, selector_year_b1, selector_year_b2, selector_year_b8)
print("-" * 50)

Obteniendo opciones de https://www.catastro.hacienda.gob.es/esp/estadistica_1.asp...
✅ Opciones obtenidas.
Obteniendo opciones de https://www.catastro.hacienda.gob.es/esp/estadistica_4.asp...
✅ Opciones obtenidas.
Obteniendo opciones de https://www.catastro.hacienda.gob.es/esp/estadistica_6.asp...
✅ Opciones obtenidas.


Dropdown(description='Provincia:', index=13, options={'Araba/Álava': '01', 'Albacete': '02', 'Alicante': '03',…

Dropdown(description='[Urbano/Rústico/BICE]:', index=19, options={'2006': '2006', '2007': '2007', '2008': '200…

Dropdown(description='[Distrito/Uso]:', index=14, options={'2011': '2011', '2012': '2012', '2013': '2013', '20…

Dropdown(description='[IBI]:', index=18, options={'2006': '2006', '2007': '2007', '2008': '2008', '2009': '200…

--------------------------------------------------


## Celda 4: Funciones Principales de Extracción y Procesamiento

### El Motor del Scraper

Esta celda contiene el núcleo funcional del notebook: las funciones que realizan el trabajo pesado de descargar los datos y convertirlos en un formato útil.

1.  **Función `descargar_html_tabla` (con `Playwright`)**:
    - **Propósito**: Automatizar la navegación para obtener el HTML de la tabla de resultados.
    - **Funcionamiento**:
        - Abre la URL en un navegador `Chromium` invisible.
        - Selecciona **todas las opciones** en los menús de criterios (`cri1`, `cri2`).
        - Hace clic en el botón `"Consultar todo"`.
        - Detecta la nueva ventana emergente con los resultados y guarda su contenido HTML en un archivo local.
    
2.  **Función `extraer_datos_de_html` (con `BeautifulSoup`)**:
    - **Propósito**: Leer el archivo `.html` y convertir la tabla en un `DataFrame` de `pandas`.
    - **Funcionamiento**:
        - Parsea el HTML y localiza la tabla de datos principal.
        - Extrae los encabezados y las filas de datos.
        - ***Limpia los datos***: elimina espacios, quita los separadores de miles y estandariza la coma decimal a un punto, preparando los números para el análisis.

3.  **Función `extraer_datos_de_html_multinivel`**:
    - **Propósito**: Es una versión especializada para manejar tablas con **doble cabecera**, como el caso de "Tipo de cultivo" (`05b_rust_porc_tipo_cultivo`).
    - **Funcionamiento**: Lee las dos primeras filas de la tabla para construir un encabezado combinado (ej: `Cítricos_Sup_%`) antes de procesar los datos.

In [4]:
# =============================================================================
# Celda 4: Lógica principal de scraping con Playwright
# =============================================================================

async def _descargar_html_una_vez(context, url: str, ruta_destino_html: str, timeout_ms: int = 60_000) -> bool:
    """
    Realiza un único intento de descargar el HTML de una tabla de datos del Catastro.

    La función navega a una URL, selecciona todas las opciones de los filtros,
    hace clic en "Consultar todo" y guarda el HTML de la tabla resultante.
    Maneja los dos comportamientos observados del sitio web:
      1. La tabla se abre en una nueva pestaña (popup).
      2. La tabla se carga en la misma pestaña (fallback).

    Args:
        context: El contexto del navegador de Playwright.
        url: La URL de la página de consulta.
        ruta_destino_html: La ruta del archivo donde se guardará el HTML.
        timeout_ms: Tiempo máximo de espera para las operaciones en milisegundos.

    Returns:
        True si la descarga y el guardado fueron exitosos, False en caso contrario.
    """
    page = await context.new_page()  # Abre una nueva pestaña para la operación.
    try:
        await page.goto(url, timeout=timeout_ms)
        await page.wait_for_selector('select[name="cri1"]', timeout=20_000)

        # Selecciona todas las opciones del primer criterio (obligatorio).
        c1_values = await page.eval_on_selector('select[name="cri1"]', 'el => Array.from(el.options).map(o => o.value)')
        await page.select_option('select[name="cri1"]', c1_values)

        # Si existe un segundo criterio, también selecciona todas sus opciones.
        if await page.query_selector('select[name="cri2"]'):
            c2_values = await page.eval_on_selector('select[name="cri2"]', 'el => Array.from(el.options).map(o => o.value)')
            await page.select_option('select[name="cri2"]', c2_values)

        # El botón de consulta puede tener 'alt' o 'value' como atributo identificador.
        selector_boton = 'input[alt="Consultar todo"], input[value="Consultar todo"]'
        html_content = None

        # Intenta capturar la tabla de una nueva pestaña (comportamiento principal).
        try:
            async with context.expect_page(timeout=5_000) as new_page_info:
                await page.click(selector_boton)

            new_page = await new_page_info.value
            await new_page.wait_for_load_state('domcontentloaded', timeout=timeout_ms)
            html_content = await new_page.content()
            await new_page.close()
        except Exception:
            # Fallback: si no se abrió una nueva pestaña, la tabla cargó en la actual.
            await page.click(selector_boton)
            await page.wait_for_load_state('domcontentloaded', timeout=timeout_ms)
            html_content = await page.content()

        if not html_content:
            return False  # Si no se pudo obtener el contenido, el intento falla.

        # Guarda el HTML en un archivo para el procesamiento posterior.
        with open(ruta_destino_html, "w", encoding="utf-8") as f:
            f.write(html_content)

        await asyncio.sleep(0.5)  # Pausa de cortesía para no sobrecargar el servidor.
        return True

    except Exception as e:
        print(f"   ❌ Error durante la descarga con Playwright: {type(e).__name__}")
        return False
    finally:
        # Asegura que la pestaña se cierre siempre para liberar recursos.
        if not page.is_closed():
            await page.close()


async def descargar_html_tabla_con_reintentos(context, url: str, ruta_destino_html: str, timeout_ms: int = 60_000, reintentos: int = 2) -> bool:
    """
    Ejecuta la descarga de una tabla con una política de reintentos.

    Esta función envuelve a `_descargar_html_una_vez` para manejar fallos
    transitorios (de red, de la UI del sitio, etc.), reintentando la operación
    varias veces antes de desistir.

    Args:
        context: El contexto del navegador de Playwright.
        url: La URL de la página de consulta.
        ruta_destino_html: La ruta del archivo donde se guardará el HTML.
        timeout_ms: Timeout para cada intento.
        reintentos: Número de reintentos adicionales si el primer intento falla.

    Returns:
        True si la descarga fue exitosa en cualquiera de los intentos, False si todos fallaron.
    """
    for intento in range(reintentos + 1):
        if await _descargar_html_una_vez(context, url, ruta_destino_html, timeout_ms):
            return True  # Éxito.
        if intento < reintentos:
            print(f"   🔁 Reintentando ({intento + 1}/{reintentos})...")
    return False # Todos los intentos fallaron.


print("✅ Funciones de descarga y procesamiento listas.")
print("-" * 50)

✅ Funciones de descarga y procesamiento listas.
--------------------------------------------------


## Celda 5: Ejecución del Proceso Completo

### Orquestador de la Extracción

Esta es la celda que pone todo en marcha. Contiene un botón que, al ser pulsado, ejecuta el proceso completo de principio a fin.

- **Recopilar Selección del Usuario**: Lee los valores de `año` y `provincia` de los widgets.
- **Crear Directorio de Salida**: Genera una carpeta única para guardar los resultados (ej: `/content/catastro_CORDOBA_2025_2024_2024`).
- **Generar todas las URLs**: Llama a las funciones de la Celda 2 para crear la lista completa de archivos a descargar.
- **Iniciar el Bucle de Extracción**:
    - Itera sobre cada URL.
    - Llama a `descargar_html_tabla` para obtener el `.html`.
    - **Lógica Condicional**: Elige la función de *parsing* correcta según el archivo. Usa `extraer_datos_de_html_multinivel` para el caso especial y `extraer_datos_de_html` para el resto.
    - Guarda el resultado como un archivo `.csv` con **punto y coma (`;`)** como separador para asegurar la compatibilidad con Excel en español.
    - Elimina el archivo `.html` intermedio.*texto en cursiva*

In [5]:
# =============================================================================
# Celda 5: Parsers de HTML a DataFrame
# (Incluye detección de "sin datos", desambiguación de cabeceras y limpieza numérica)
# =============================================================================

def _clean_num(txt: str) -> str:
    """Normaliza un string numérico para su posterior conversión.

    Elimina espacios indivisibles (nbsp), separadores de miles ('.') y
    reemplaza la coma decimal por un punto.
    Ej: '1.234,56\xa0' -> '1234.56'
    """
    return txt.replace('\xa0', '').replace('.', '').replace(',', '.')

def _dedupe_headers(cols: list) -> list:
    """Renombra cabeceras duplicadas añadiendo un sufijo numérico.

    Ej: ['A', 'B', 'A', 'C', 'A'] -> ['A', 'B', 'A_2', 'C', 'A_3']
    """
    seen = {}
    out = []
    for c in cols:
        if c not in seen:
            seen[c] = 1
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")
    return out

def extraer_datos_de_html(path_html: str, nombre_primera_columna: str = "Entidad") -> Optional[pd.DataFrame]:
    """
    Parsea un archivo HTML que contiene una tabla con una sola fila de cabecera.

    Args:
        path_html: Ruta al archivo HTML guardado.
        nombre_primera_columna: Nombre a asignar a la primera columna de la tabla.

    Returns:
        Un DataFrame de pandas con los datos extraídos, o None si no hay datos
        o no se puede parsear la tabla.
    """
    try:
        with open(path_html, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "lxml")
    except FileNotFoundError:
        print(f"⚠️  El archivo {path_html} no fue encontrado para procesar.")
        return None

    # Si la página contiene un mensaje explícito de "no hay datos", se detiene el proceso.
    if soup.find(string=re.compile("No hay datos|no se dispone de información", re.IGNORECASE)):
        return None

    # Busca la tabla de datos de forma robusta a partir de una celda de cabecera conocida.
    header_cell = soup.find("td", class_="tableCellGr")
    if not header_cell: return None
    tabla = header_cell.find_parent("table")
    if not tabla: return None
    header_row = tabla.find("tr")
    if not header_row: return None

    # Extrae, renombra y desambigua las cabeceras.
    headers = [th.get_text(strip=True) for th in header_row.find_all("td", class_="tableCellGr")]
    if not headers: return None
    headers[0] = nombre_primera_columna
    headers = _dedupe_headers(headers)

    # Extrae las filas de datos, limpiando los valores numéricos.
    data_rows = []
    for row in header_row.find_next_siblings("tr"):
        row_data = []
        first_cell = row.find("td", class_="tableCellGr")
        if first_cell:
            row_data.append(first_cell.get_text(strip=True))

        data_cells = row.find_all("td", class_="dataCell")
        for cell in data_cells:
            row_data.append(_clean_num(cell.get_text(strip=True)))

        if len(row_data) == len(headers):
            data_rows.append(row_data)

    if not data_rows: return None

    df = pd.DataFrame(data_rows, columns=headers)

    # Convierte las columnas de datos a tipo numérico, manejando errores.
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def extraer_datos_de_html_multinivel(path_html: str, nombre_primera_columna: str = "Municipio") -> Optional[pd.DataFrame]:
    """
    Parsea un archivo HTML que contiene una tabla con cabecera de dos niveles y 'colspan'.

    Args:
        path_html: Ruta al archivo HTML guardado.
        nombre_primera_columna: Nombre a asignar a la primera columna de la tabla.

    Returns:
        Un DataFrame de pandas con los datos, o None si no hay datos.
    """
    try:
        with open(path_html, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "lxml")
    except FileNotFoundError:
        print(f"⚠️  El archivo {path_html} no fue encontrado para procesar.")
        return None

    if soup.find(string=re.compile("No hay datos|no se dispone de información", re.IGNORECASE)):
        return None

    header_cell = soup.find("td", class_="tableCellGr")
    if not header_cell: return None
    tabla = header_cell.find_parent("table")
    if not tabla: return None
    rows = tabla.find_all("tr")
    if len(rows) < 3: return None  # Necesita 2 filas de cabecera y al menos 1 de datos.

    # Procesa la primera fila de cabecera, expandiendo las celdas con 'colspan'.
    h1 = []
    for td in rows[0].find_all("td", class_="tableCellGr")[1:]:
        text = td.get_text(strip=True).replace(' ', '_')
        colspan = int(td.get('colspan', 1))
        h1.extend([text] * colspan)

    # Procesa la segunda fila de cabecera (sub-etiquetas).
    map_h2 = {'Superficie rústica': 'Sup_%', 'Subparcelas': 'Subparc_%', 'Valor catastral': 'ValorCat_%'}
    h2_cells = rows[1].find_all("td")[1:]
    h2 = [map_h2.get(td.get_text(strip=True), td.get_text(strip=True)) for td in h2_cells]

    # Combina ambas cabeceras y desambigua los nombres finales.
    headers = [nombre_primera_columna] + [f"{a}_{b}" for a, b in zip(h1, h2)]
    headers = _dedupe_headers(headers)

    # Extrae las filas de datos.
    data = []
    for row in rows[2:]:
        cells = [td.get_text(strip=True) for td in row.find_all("td", class_="tableCellGr")]
        cells += [_clean_num(td.get_text(strip=True)) for td in row.find_all("td", class_="dataCell")]
        if len(cells) == len(headers):
            data.append(cells)

    if not data: return None

    df = pd.DataFrame(data, columns=headers)
    for col in df.columns[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [6]:
# =============================================================================
# Celda 6: Orquestador del Proceso de Extracción
# (Incluye barra de progreso, logging, reanudación y filtro de subconjuntos)
# =============================================================================

# --- Widgets de Interfaz de Usuario ---
start_button = widgets.Button(description="Iniciar Extracción Completa", button_style='success')
output_area = widgets.Output()
display(start_button, output_area)

from ipywidgets import IntProgress # Importación específica para la barra de progreso.

# --- Funciones Auxiliares ---

def _year_for(nombre_stat: str, y_b1: str, y_b2: str, y_b8: str) -> str:
    """Devuelve el año correcto para un fichero según el bloque al que pertenece."""
    if nombre_stat.startswith("02"): return y_b2
    if nombre_stat.startswith("08"): return y_b8
    return y_b1

def _sanitize(text: str) -> str:
    """Normaliza un texto para que sea seguro usarlo en nombres de archivo o rutas.

    Elimina acentos y reemplaza cualquier caracter no alfanumérico por guiones bajos.
    """
    no_accents = "".join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))
    return re.sub(r'[^A-Za-z0-9_]+', '_', no_accents).strip('_')


# --- Orquestador Principal ---

def on_button_clicked(b):
    """
    Controlador (handler) que se activa al hacer clic en el botón de inicio.

    Limpia el área de salida y lanza la tarea principal de extracción asíncrona.
    """
    with output_area:
        clear_output(wait=True)
        # La ejecución real ocurre en la función asíncrona 'run_extraction'.
        # asyncio.run() es el puente que la inicia desde este entorno síncrono.
        asyncio.run(run_extraction())

async def run_extraction():
    """
    Función principal asíncrona que orquesta todo el proceso de scraping y guardado.

    Pasos:
    1.  Recoge los parámetros de los widgets.
    2.  Configura el directorio de salida y el logging.
    3.  Construye la lista de todas las URLs a procesar.
    4.  (Opcional) Filtra la lista para procesar solo un subconjunto.
    5.  Inicia el navegador Playwright.
    6.  Itera sobre cada URL:
        a. Comprueba si el resultado ya existe (reanudación).
        b. Descarga el HTML de la tabla con reintentos.
        c. Parsea el HTML con el extractor adecuado.
        d. Guarda los datos como un archivo CSV.
        e. Limpia los archivos temporales.
    7.  Cierra el navegador y muestra un resumen.
    """
    # 1. Recoger parámetros de la UI
    anio_b1 = selector_year_b1.value
    anio_b2 = selector_year_b2.value
    anio_b8 = selector_year_b8.value
    cod_prov = selector_provincia.value
    nombre_prov = [k for k, v in opciones_provincia.items() if v == cod_prov][0]

    # 2. Configurar entorno de salida
    nombre_prov_norm = _sanitize(nombre_prov)
    output_dir = f"/content/catastro_{nombre_prov_norm}_{anio_b1}_{anio_b2}_{anio_b8}"
    os.makedirs(output_dir, exist_ok=True)
    log_path = os.path.join(output_dir, "proceso.log")

    def log(msg: str):
        print(msg)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(msg + "\n")

    log(f"📂 Directorio de resultados: {output_dir}")

    # 3. Construir lista de tareas (URLs)
    cod_prov_2d = _prov(cod_prov)
    all_urls = {
        **build_urls_simple_prefix(int(anio_b1), cod_prov_2d, BASE_B1, PREFIJOS_B1),
        **build_urls_b2(int(anio_b2), cod_prov_2d),
        **build_urls_simple_prefix(int(anio_b1), cod_prov_2d, BASE_B4, PREFIJOS_B4),
        **build_urls_simple_prefix(int(anio_b1), cod_prov_2d, BASE_B5, PREFIJOS_B5),
        **build_urls_simple_prefix(int(anio_b8), cod_prov_2d, BASE_B8, PREFIJOS_B8),
        **build_urls_b9(int(anio_b1), cod_prov_2d),
        **build_urls_b12(int(anio_b1), cod_prov_2d)
    }

    # 4. (Opcional) Filtrar tareas para una ejecución parcial
    SUBSET_PREFIX = None  # Cambiar a "01", "02", etc., para ejecutar solo un bloque.
    if SUBSET_PREFIX:
        all_urls = {k: v for k, v in all_urls.items() if k.startswith(SUBSET_PREFIX)}

    orden_procesamiento = sorted(all_urls.keys())
    pbar = IntProgress(min=0, max=len(orden_procesamiento), description='Progreso')
    display(pbar)

    # 5. Iniciar el navegador
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent=(
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0.0.0 Safari/537.36"
        ))

        ok_count, fail_count = 0, 0

        # 6. Bucle principal de procesamiento
        for nombre_stat in orden_procesamiento:
            url = all_urls[nombre_stat]
            year_for_file = _year_for(nombre_stat, anio_b1, anio_b2, anio_b8)
            ruta_html = os.path.join(output_dir, f"{nombre_stat}.html")
            ruta_csv = os.path.join(output_dir, f"{year_for_file}_{cod_prov_2d}_{nombre_stat}.csv")

            log(f"\n- Procesando: {nombre_stat}")

            # 6.a. Reanudación
            if os.path.exists(ruta_csv):
                log(f"   ⏭️  Ya existe {os.path.basename(ruta_csv)}; saltando.")
                pbar.value += 1
                continue

            # 6.b. Descarga
            success = await descargar_html_tabla_con_reintentos(context, url, ruta_html)
            if not success:
                log("   ❌ Descarga fallida.")
                fail_count += 1
                pbar.value += 1
                continue

            # 6.c. Parseo
            log("   ✅ HTML guardado. Parseando...")
            # La tabla de cultivos tiene una estructura de cabecera especial (multinivel).
            if nombre_stat == "05b_rust_porc_tipo_cultivo":
                df = extraer_datos_de_html_multinivel(ruta_html, nombre_primera_columna="Municipio")
            else:
                nombre_col = "Distrito" if nombre_stat.startswith("02") else "Municipio"
                df = extraer_datos_de_html(ruta_html, nombre_primera_columna=nombre_col)

            # 6.d. Guardado
            if df is not None and not df.empty:
                df.to_csv(ruta_csv, index=False, sep=';', encoding='utf-8-sig')
                log(f"   💾 CSV: {os.path.basename(ruta_csv)}")
                ok_count += 1
            else:
                log("   ⚠️ Tabla no válida o sin datos.")
                fail_count += 1

            # 6.e. Limpieza
            try:
                os.remove(ruta_html)
            except OSError:
                pass

            pbar.value += 1

        # 7. Cierre y resumen
        await browser.close()
        log("\n" + "-"*50)
        log(f"¡Completado! ✔️ {ok_count} correctos | ❌ {fail_count} fallidos.")
        print(f"📝 Log de la sesión guardado en: {log_path}")

# Asocia la función controladora al evento 'on_click' del botón.
start_button.on_click(on_button_clicked)

Button(button_style='success', description='Iniciar Extracción Completa', style=ButtonStyle())

Output()

## Celda 7: Comprimir y Descargar Resultados

### Empaquetado y Entrega

Una vez finalizada la extracción, esta celda proporciona una forma cómoda de descargar todos los ficheros `.csv` juntos.

- **Botón `"Comprimir y Descargar"`**: Inicia la acción de empaquetado.
- **Lógica de Compresión**:
    - **Normaliza el nombre** de la provincia (mayúsculas, sin acentos) para evitar problemas de compatibilidad entre sistemas operativos.
    - Utiliza la librería `shutil` para crear un **archivo `.zip`** que contiene todos los ficheros generados.
    - Nombra el archivo siguiendo el formato `CODPROV_NOMBREPROV.zip` (ej: `14_CORDOBA.zip`).
- **Descarga**: Inicia automáticamente la descarga del archivo `.zip` en el navegador del usuario.

In [7]:
# =============================================================================
# Celda 7: Comprimir y Descargar Resultados
# =============================================================================

# --- Widgets de Interfaz de Usuario ---
download_button = widgets.Button(description="Comprimir y Descargar", button_style='info')
download_output_area = widgets.Output()
display(download_button, download_output_area)

# --- Lógica de Compresión y Descarga ---

def on_download_clicked(b):
    """
    Controlador para el botón de descarga.

    Localiza la carpeta de resultados generada en la celda 6, la comprime
    en un archivo .zip y, si se ejecuta en Google Colab, inicia la descarga
    en el navegador.

    Incluye una lógica de fallback robusta para encontrar la carpeta de
    resultados incluso si su nombre no coincide exactamente con el esperado,
    probando con formatos de nombre antiguos o buscando patrones genéricos.
    """
    with download_output_area:
        clear_output(wait=True)

        # Reconstruye la ruta del directorio de resultados basándose en los widgets.
        # Es crucial usar la misma lógica de la celda 6 para encontrar la carpeta correcta.
        anio_b1 = selector_year_b1.value
        anio_b2 = selector_year_b2.value
        anio_b8 = selector_year_b8.value
        cod_prov = selector_provincia.value
        nombre_prov = [k for k, v in opciones_provincia.items() if v == cod_prov][0]

        nombre_prov_norm = _sanitize(nombre_prov)
        output_dir = f"/content/catastro_{nombre_prov_norm}_{anio_b1}_{anio_b2}_{anio_b8}"

        # Lógica de fallback: si la carpeta esperada no existe, intenta encontrarla.
        if not os.path.isdir(output_dir):
            print(f"Buscando carpeta de resultados (no se encontró en la ruta esperada)...")
            found_dir = None

            # Estrategia 1: Probar con un formato de nombre antiguo (sin normalización de acentos).
            nombre_prov_norm_old = re.sub(r'[^A-Za-z0-9_]+', '_', nombre_prov)
            alt_dir_1 = f"/content/catastro_{nombre_prov_norm_old}_{anio_b1}_{anio_b2}_{anio_b8}"
            if os.path.isdir(alt_dir_1):
                found_dir = alt_dir_1

            # Estrategia 2: Si la anterior falla, buscar con un patrón de comodín.
            if not found_dir:
                import glob
                pattern = f"/content/catastro_*_{anio_b1}_{anio_b2}_{anio_b8}"
                matches = [d for d in glob.glob(pattern) if os.path.isdir(d)]
                if matches:
                    found_dir = matches[0] # Usa la primera coincidencia.

            if found_dir:
                print(f"ℹ️ Se usará la carpeta encontrada: {found_dir}")
                output_dir = found_dir
            else:
                print(f"❌ Error: No se encontró el directorio de resultados '{output_dir}'.")
                print("   Asegúrate de haber ejecutado la celda de extracción (Celda 6) primero.")
                return

        # Crea un nombre de archivo para el ZIP (ej: '14_CORDOBA.zip').
        nombre_sin_acentos = "".join(c for c in unicodedata.normalize('NFKD', nombre_prov) if not unicodedata.combining(c))
        nombre_prov_zip = re.sub(r'[^A-Za-z0-9_]+', '_', nombre_sin_acentos).upper()
        nombre_zip = f"{_prov(cod_prov)}_{nombre_prov_zip}"
        ruta_zip_base = f"/content/{nombre_zip}" # shutil añade la extensión .zip

        # Comprime el directorio.
        print(f"\nComprimiendo '{output_dir}' en '{nombre_zip}.zip'...")
        try:
            shutil.make_archive(ruta_zip_base, 'zip', output_dir)
            print(f"✅ Compresión finalizada.")
        except Exception as e:
            print(f"❌ Error durante la compresión: {e}")
            return

        # Si se ejecuta en Colab (donde 'files' está disponible), inicia la descarga.
        if files:
            print("Iniciando descarga en el navegador...")
            files.download(f"{ruta_zip_base}.zip")

# Asocia el controlador al evento 'on_click' del botón.
download_button.on_click(on_download_clicked)


Button(button_style='info', description='Comprimir y Descargar', style=ButtonStyle())

Output()

## Celda 8: Limpieza del Entorno

### Mantenimiento y Orden

Esta última celda es una utilidad de "limpieza" que permite restablecer el entorno sin tener que reiniciar todo el notebook.

- **Botón `"Limpiar Resultados Anteriores"`**: Ejecuta el borrado.
- **Lógica de Borrado**:
    - Determina los nombres del directorio de resultados y del archivo `.zip`.
    - Comprueba si existen y, si es así, los **elimina por completo** (`shutil.rmtree` y `os.remove`).
    - Muestra un mensaje informativo del resultado de la operación.

> *Consejo: Esta celda es muy útil si quieres realizar varias extracciones seguidas para diferentes provincias, ya que mantiene el espacio de trabajo limpio y organizado.*

In [8]:
# =============================================================================
# Celda 8: Limpieza de Resultados Anteriores
# =============================================================================

# --- Widgets de Interfaz de Usuario ---
cleanup_button = widgets.Button(description="Limpiar Resultados Anteriores", button_style='danger')
cleanup_output_area = widgets.Output()
display(cleanup_button, cleanup_output_area)

# --- Lógica de Limpieza ---

def on_cleanup_clicked(b):
    """
    Controlador para el botón de limpieza.

    Localiza y elimina la carpeta de resultados y el archivo .zip asociados
    a la configuración actual de los widgets. Utiliza la misma lógica de
    búsqueda (incluyendo fallbacks) que las celdas de compresión para
    asegurar que se encuentran los objetivos correctos antes de eliminarlos.
    """
    with cleanup_output_area:
        clear_output(wait=True)

        # 1. Reconstruir las rutas de los artefactos a eliminar.
        # Es crucial usar la misma lógica que en las celdas 6 y 7.
        anio_b1 = selector_year_b1.value
        anio_b2 = selector_year_b2.value
        anio_b8 = selector_year_b8.value
        cod_prov = selector_provincia.value
        nombre_prov = [k for k, v in opciones_provincia.items() if v == cod_prov][0]

        # 2. Localizar el directorio de resultados con la misma lógica de fallbacks.
        nombre_prov_norm = _sanitize(nombre_prov)
        output_dir_to_delete = f"/content/catastro_{nombre_prov_norm}_{anio_b1}_{anio_b2}_{anio_b8}"

        if not os.path.isdir(output_dir_to_delete):
            found_dir = None
            # Estrategia 1: Formato de nombre antiguo.
            nombre_prov_norm_old = re.sub(r'[^A-Za-z0-9_]+', '_', nombre_prov)
            alt_dir_1 = f"/content/catastro_{nombre_prov_norm_old}_{anio_b1}_{anio_b2}_{anio_b8}"
            if os.path.isdir(alt_dir_1):
                found_dir = alt_dir_1
            # Estrategia 2: Búsqueda con comodín.
            elif not found_dir:
                import glob
                pattern = f"/content/catastro_*_{anio_b1}_{anio_b2}_{anio_b8}"
                matches = [d for d in glob.glob(pattern) if os.path.isdir(d)]
                if matches:
                    found_dir = matches[0]

            if found_dir:
                output_dir_to_delete = found_dir

        # 3. Reconstruir la ruta del archivo ZIP de forma coherente.
        nombre_sin_acentos = "".join(c for c in unicodedata.normalize('NFKD', nombre_prov) if not unicodedata.combining(c))
        nombre_prov_zip = re.sub(r'[^A-Za-z0-9_]+', '_', nombre_sin_acentos).upper()
        zip_path_to_delete = f"/content/{_prov(cod_prov)}_{nombre_prov_zip}.zip"

        print("Iniciando limpieza...")

        # 4. Eliminar el directorio si se encontró.
        if os.path.isdir(output_dir_to_delete):
            try:
                shutil.rmtree(output_dir_to_delete)
                print(f"✅ Directorio eliminado: {output_dir_to_delete}")
            except OSError as e:
                print(f"❌ Error al eliminar el directorio: {e}")
        else:
            print(f"ℹ️ El directorio no existe (nada que limpiar): {output_dir_to_delete}")

        # 5. Eliminar el archivo ZIP si existe.
        if os.path.exists(zip_path_to_delete):
            try:
                os.remove(zip_path_to_delete)
                print(f"✅ Archivo ZIP eliminado: {zip_path_to_delete}")
            except OSError as e:
                 print(f"❌ Error al eliminar el ZIP: {e}")
        else:
            print(f"ℹ️ El archivo ZIP no existe (nada que limpiar): {zip_path_to_delete}")

        print("\nLimpieza completada.")

# Asocia el controlador al evento 'on_click' del botón de limpieza.
cleanup_button.on_click(on_cleanup_clicked)



Button(button_style='danger', description='Limpiar Resultados Anteriores', style=ButtonStyle())

Output()